In [76]:
import cudaq

######################### JENGA ##################################
def pce_loss_function(params):
    # 1. Measure all 60 Pauli expectation values

    
    @cudaq.kernel
    def qc(n: int, variable_list: list[float], pauli: list[int], t: int):
        qubits = cudaq.qvector(n)
        
        for i in range(n):
            h(qubits[i])
    
        for l in range(len(variable_list)//15):
    
            for i in range(n):
                ry(variable_list[15*l+i], qubits[i])
                rz(variable_list[15*l+5+i], qubits[i])
                
                h(qubits[i])
                
            for i in range(n):
                if i != n-1:
                    cx(qubits[i], qubits[i+1])
                    rz(variable_list[15*l+10+i], qubits[i+1])
                    cx(qubits[i], qubits[i+1])
                else:
                    cx(qubits[i], qubits[0])
                    rz(variable_list[15*l+10+i], qubits[i])
                    cx(qubits[i], qubits[0])
                
    
        for i in range(n):
            if pauli[n*t+i] == 0:
                h(qubits[i])
            elif pauli[n*t+i] == 1:
                sdg(qubits[i])
                h(qubits[i])
                
        mz(qubits)
    
    n = 60 
    num_qubits = 5
    shots_var = 100000
    runs = 1
    
    variables = params
    
    string_list = []
    data = []
    for t in range(len(pauli_list)):
        result = cudaq.sample(qc, num_qubits, variables, pauli_flat, t, shots_count=shots_var)
        data += [list(result.items())]
    
    
    E_list = []
    for i in range(len(pauli_list)):
        E = 0
        mult = 1
        for w in range(len(data[i])):
            for j in data[i][w][0]:
                if j == "0":
                    mult *= -1
            
            E += mult*data[i][w][1]/shots_var
        E_list += [E]

    
    # 2. Apply tanh relaxation

    
    alpha = 6.0
    beta = 15.0
    x_tilde = [np.tanh(alpha * exp) for exp in E_list]
    
    
    # 3. Compute LABS loss
    N = len(x_tilde)
    loss = 0.0

    # Autocorrelation energy term
    for ell in range(1, N):
        for i in range(N - ell):
            loss += (x_tilde[i] * x_tilde[i+ell])**2
    
    # Regularization term
    loss -= beta * sum(x**2 for x in x_tilde)
    
    return loss

In [80]:
import numpy as np

a = pce_loss_function([1,2,1,1,1, 2,1,2,2,2, 3,2,3,3,3, 12,11,12,12,12, 22,21,22,22,22, 32,31,32,32, 32])

print(a)

0.03247268132571435


In [35]:
import random

def random_pauli_string(n):
    paulis = ['X', 'Y', 'Z']
    return ''.join(random.choice(paulis) for _ in range(n))


pauli_list = []
while len(pauli_list) != n:
    pauli = random_pauli_string(5)
    if pauli in pauli_list:
        3
    else:
        pauli_list += [pauli]

pauli_flat = []
for i in pauli_list:
    for j in i:
        if j == "X":
            pauli_flat += [0]
        elif j == "Y":
            pauli_flat += [1]
        else:
            pauli_flat += [2]

print(pauli_list)
print(pauli_flat)

['YZZXX', 'YYZXY', 'XZYXY', 'ZXXXX', 'XZZZZ', 'YXXYZ', 'ZZXXY', 'XZXXY', 'ZYZZZ', 'ZXXYZ', 'ZYYXY', 'YXXXY', 'YXZYY', 'ZZYXZ', 'XZYYZ', 'YZYXX', 'YXYXX', 'ZZZZZ', 'YYYZZ', 'YYYXX', 'XXXYZ', 'XXZYX', 'YZYZZ', 'ZXZZX', 'XXXZZ', 'ZXYYX', 'ZYXZY', 'XYYZZ', 'YYZYX', 'XZYYY', 'YYXYX', 'XZXXZ', 'XXYZX', 'XZXZX', 'ZZXXZ', 'YZZYX', 'ZZXZY', 'ZZXYY', 'ZYXZZ', 'YYYXZ', 'XYZZZ', 'XXZZX', 'XZXYX', 'YYYZY', 'YZYXZ', 'ZYYZY', 'YZXZX', 'ZZYZZ', 'YZYYX', 'YXYXY', 'ZZZYZ', 'XYZYY', 'XYXXX', 'ZYXYY', 'XYYZX', 'YZYXY', 'XXYYY', 'XXZZZ', 'ZZXZX', 'YYXZZ']
[1, 2, 2, 0, 0, 1, 1, 2, 0, 1, 0, 2, 1, 0, 1, 2, 0, 0, 0, 0, 0, 2, 2, 2, 2, 1, 0, 0, 1, 2, 2, 2, 0, 0, 1, 0, 2, 0, 0, 1, 2, 1, 2, 2, 2, 2, 0, 0, 1, 2, 2, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 2, 1, 1, 2, 2, 1, 0, 2, 0, 2, 1, 1, 2, 1, 2, 1, 0, 0, 1, 0, 1, 0, 0, 2, 2, 2, 2, 2, 1, 1, 1, 2, 2, 1, 1, 1, 0, 0, 0, 0, 0, 1, 2, 0, 0, 2, 1, 0, 1, 2, 1, 2, 2, 2, 0, 2, 2, 0, 0, 0, 0, 2, 2, 2, 0, 1, 1, 0, 2, 1, 0, 2, 1, 0, 1, 1, 2, 2, 1, 1, 2, 1, 0, 0, 2, 1, 1, 1, 1, 1, 0,